### Model Analysis tutorial ###

#### Load model and evaluate on some test data #####

In [ ]:
%%capture
import os

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import SimpleITK as sitk
import torch
from monai.data import DataLoader, Dataset
from monai.transforms import (
    AsDiscreted,
    Compose,
    EnsureChannelFirstd,
    LoadImaged,
    NormalizeIntensityd,
)

import prognosais.IO.constants as constants
import prognosais.model.development.utils as model_utils
from prognosais.IO.dataset import DataGenerator
from prognosais.model.architectures.CSNet import CSNet
from prognosais.model.architectures.Evaluator import Evaluator

print("CUDA available:", torch.cuda.is_available())

model_file_dir = "/gpfs/work1/0/prjs0971/PrognosAIs/experiments/060925/baseline_af1/results/models/best_model.pt"
results_dir = "/gpfs/work1/0/prjs0971/PrognosAIs/experiments/060925/baseline_af1/results"
data_dir = "/gpfs/work1/0/prjs0971/PrognosAIs/data/test"
labels_file_dir = "/gpfs/work1/0/prjs0971/PrognosAIs/data/test/labels.txt"
file_extension = constants.DATA_NIFTI_EXTENSION
data_type = "preprocessed"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
subset = 0.05
missing_value = -1
seed = 42
batch_size = 4
modalities = constants.SCAN_TYPES
mask_file_name = constants.MASK_FILE_NAME
preprocessed_folder = constants.GLIOSEG_PREPROCESSED_FOLDER
data_folders = {
    "original": constants.NIFTI_FOLDER,
    "registered": constants.REGISTERED_FOLDER,
    "preprocessed": preprocessed_folder,
}
labels_config = {
    "case_id_column": "Image",
    "tasks": {
        constants.LABEL_TASK_IDH: {
            "column": constants.CLASSIFICATION_DISPLAY_NAMES[constants.TASK_IDH],
            "output_key": "label_idh",
            "class_values": [0, 1],
            "shift_to_zero": False,
        },
        constants.LABEL_TASK_1P19Q: {
            "column": constants.CLASSIFICATION_DISPLAY_NAMES[constants.TASK_1P19Q],
            "output_key": "label_1p19q",
            "class_values": [0, 1],
            "shift_to_zero": False,
        },
        constants.LABEL_TASK_GRADE: {
            "column": constants.CLASSIFICATION_DISPLAY_NAMES[constants.TASK_GRADE],
            "output_key": "label_grade",
            "class_values": [2, 3, 4],
            "shift_to_zero": True,
        },
    },
}

if not torch.cuda.is_available():
    model_info = torch.load(model_file_dir, map_location=device)
else:
    model_info = torch.load(model_file_dir)

model = CSNet(dropout_rate=0.25, in_channels=len(modalities))
model.load_state_dict(model_info["model_state_dict"])
model.to(device)

test_dict = DataGenerator(
    data_dir=data_dir,
    file_extension=file_extension,
    labels_file_dir=labels_file_dir,
    data_type=data_type,
    train=False,
    subset=subset,
    missing_value=missing_value,
    ids_to_exclude=None,
    seed=seed,
    modalities=modalities,
    mask_file_name=mask_file_name,
    data_folders=data_folders,
    labels_config=labels_config,
)


test_transforms = Compose(
    [
        LoadImaged(keys=["image", "mask"]),
        EnsureChannelFirstd(keys=["image", "mask"]),
        NormalizeIntensityd(keys=["image"], channel_wise=True),
        AsDiscreted(keys=["mask"], to_onehot=2),
    ]
)

post_transforms = Compose(
    [
        AsDiscreted(keys=["pred_seg", "mask"], argmax=True),
    ]
)

test_dataset = Dataset(test_dict.data, transform=test_transforms)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model_evaluator = Evaluator(
    model=model,
    test_loader=test_data_loader,
    device=device,
    post_transforms=post_transforms,
    results_dir=results_dir,
)

model_evaluator.evaluate(save_predictions=False, save_activation_maps=True)


#### Get the activation maps for all test data in a single dictionary ####

In [ ]:
# Example: batch 0
model_evaluator.activation_maps[0]["enc1.0"].shape

In [ ]:
# Dataset length
print(len(test_dataset))
# Get activation maps in a single dictionary
activation_maps = model_utils.merge_activation_maps(model_evaluator.activation_maps)
print(activation_maps.keys()) # keys are the layer names
# Now we have all activation maps in a single dictionary
print(activation_maps[list(activation_maps.keys())[0]].shape)

#### Visualization of one case ####

In [ ]:
cases_id = [case["mask"].split("/")[-3] for case in test_dict]
example_case_id = cases_id[0] # Choose the first case ID for the example

scan_dir = os.path.join(data_dir, example_case_id, preprocessed_folder)
scan_dir_t1 = os.path.join(scan_dir, "T1.nii.gz")
scan_dir_t1ce = os.path.join(scan_dir, "T1CE.nii.gz")
scan_dir_t2 = os.path.join(scan_dir, "T2.nii.gz")
scan_dir_flair = os.path.join(scan_dir, "FLAIR.nii.gz")

scan_t1 = sitk.ReadImage(scan_dir_t1)
scan_t1ce = sitk.ReadImage(scan_dir_t1ce)
scan_t2 = sitk.ReadImage(scan_dir_t2)
scan_flair = sitk.ReadImage(scan_dir_flair)

slice_number = 76


scan_t1_views = model_utils.get_scan_slice_all_views(scan=scan_dir_t1,
library = "SimpleITK",
slice_number = slice_number)
scan_t1ce_views = model_utils.get_scan_slice_all_views(scan=scan_dir_t1ce,
library = "SimpleITK",
slice_number = slice_number)
scan_t2_views = model_utils.get_scan_slice_all_views(scan=scan_dir_t2,
library = "SimpleITK",
slice_number = slice_number)
scan_flair_views = model_utils.get_scan_slice_all_views(scan=scan_dir_flair,
library = "SimpleITK",
slice_number = slice_number)

model_utils.plot_mri_modalities_all_views(
    scan_t1_views, scan_t1ce_views, scan_t2_views, scan_flair_views, example_case_id
)

#### Analysis of output shapes at each level of the network ####

In [ ]:
for keys in activation_maps.keys():
    print(keys, activation_maps[keys].shape)

#### Finding mapping between original slice and downsampled slice ####

In [ ]:
layer_0 = model_utils.prepare_layer_for_visualization(activation_maps["enc1.0"])
layer_1 = model_utils.prepare_layer_for_visualization(activation_maps["enc1_2.0"])

# For the sagittal view, finding the equivalent slice in the layer 1
slice_layer_1_sagittal = model_utils.map_slice_index(slice_num = slice_number, 
original_dim = layer_0.shape[0],
downsampled_dim = layer_1.shape[0])

# For the coronal view, finding the equivalent slice in the layer 1
slice_layer_1_coronal = model_utils.map_slice_index(slice_num = slice_number, 
original_dim = layer_0.shape[1],
downsampled_dim = layer_1.shape[1])

# For the axial view, finding the equivalent slice in the layer 1 
slice_layer_1_axial = model_utils.map_slice_index(slice_num = slice_number, 
original_dim = layer_0.shape[2],
downsampled_dim = layer_1.shape[2])

print("Sagittal view - original slice:", slice_number, "mapped slice in layer 1:", slice_layer_1_sagittal)
print("Coronal view - original slice:", slice_number, "mapped slice in layer 1:", slice_layer_1_coronal)
print("Axial view - original slice:", slice_number, "mapped slice in layer 1:", slice_layer_1_axial)


#### Visualization of layer 0 filters output for the chosen patient in different orientations ###

In [ ]:
model_utils.plot_feature_maps_grid(
    feature_map=layer_0, 
    patient_index=0,
    slice_index=slice_number, 
    view='axial',
    cmap='gray', 
    title= f"Patient {cases_id[0]} - Axial view - Slice {slice_number}")

In [ ]:
model_utils.plot_feature_maps_grid(
    feature_map=layer_0, 
    patient_index=0,
    slice_index=slice_number, 
    view='sagittal',
    cmap='gray', 
    title= f"Patient {cases_id[0]} - Sagittal view - Slice {slice_number}")

In [ ]:
model_utils.plot_feature_maps_grid(
    feature_map=layer_0, 
    patient_index=0,
    slice_index=slice_number, 
    view='coronal',
    cmap='gray', 
    title= f"Patient {cases_id[0]} - Coronal view - Slice {slice_number}")

#### Visualization of layer 1 filters output for the chosen patient in different orientations ###

In [ ]:
model_utils.plot_feature_maps_grid(
    feature_map=layer_1, 
    patient_index=0,
    slice_index=slice_layer_1_axial, 
    view='axial',
    cmap='gray', 
    title= f"Patient {cases_id[0]} - Axial view - Slice {slice_layer_1_axial}")

In [ ]:
model_utils.plot_feature_maps_grid(
    feature_map=layer_1, 
    patient_index=0,
    slice_index=slice_layer_1_sagittal, 
    view='sagittal',
    cmap='gray', 
    title= f"Patient {cases_id[0]} - Sagittal view - Slice {slice_layer_1_sagittal}")

In [ ]:
model_utils.plot_feature_maps_grid(
    feature_map=layer_1, 
    patient_index=0,
    slice_index=slice_layer_1_coronal, 
    view='coronal',
    cmap='gray', 
    title= f"Patient {cases_id[0]} - Coronal view - Slice {slice_layer_1_coronal}")